# PicoRV32 UART Monitor — PC 端 (无 pyserial)

**运行位置**: 宿主机 PC (Linux / Mac)  
**假设**: pyserial **不可用**，使用 `stty` + `cat` / `dd` 读取串口

## 工作流程

1. 配置串口设备 (`stty`)
2. 用 `cat` / `dd` 读取 UART 数据并保存到文件
3. 解析推理结果
4. 与 golden results 对比
5. (可选) 将结果传回 PYNQ 进行交叉验证

## 硬件连接

USB-TTL 适配器的 USB 端插入 **PC**:

```
PYNQ-Z2 PMODA              USB-TTL                PC
  Pin 1 (Y18, TX) -------> RXD                     |
  Pin 5 (GND)     -------> GND    ----USB---->  /dev/ttyUSB0 (Linux)
                                                 /dev/cu.usbserial-xxx (Mac)
                                                 (Windows: 请使用 PuTTY)
```

## 配合 PYNQ 端 notebook

同时在 PYNQ 上运行 `picorv32_mlp_pynq_no_pyserial.ipynb`:
- PYNQ 端: 加载 bitstream + 用 small_mlp_data 做 golden 推理
- PC 端 (本 notebook): 读取 UART + 解析 + 对比

## 1. 检测串口设备

In [ ]:
import os, sys, time, glob, platform, subprocess
import numpy as np

system = platform.system()
print(f"OS: {system}")
print()

if system == "Linux":
    patterns = ["/dev/ttyUSB*", "/dev/ttyACM*"]
    serial_ports = []
    for pat in patterns:
        serial_ports.extend(glob.glob(pat))
    print("Available serial ports:")
    if serial_ports:
        for p in sorted(serial_ports):
            # Show device info
            try:
                info = subprocess.check_output(
                    ["udevadm", "info", "--query=property", f"--name={p}"],
                    text=True, stderr=subprocess.DEVNULL
                )
                model = ""
                for line in info.split("\n"):
                    if "ID_MODEL=" in line:
                        model = line.split("=", 1)[1]
                print(f"  {p}  ({model})")
            except Exception:
                print(f"  {p}")
    else:
        print("  None found! Check: dmesg | tail")
        print("  Make sure USB-TTL is plugged into PC")

elif system == "Darwin":  # macOS
    patterns = ["/dev/cu.usbserial*", "/dev/cu.wchusbserial*", "/dev/cu.SLAB*"]
    serial_ports = []
    for pat in patterns:
        serial_ports.extend(glob.glob(pat))
    print("Available serial ports:")
    if serial_ports:
        for p in sorted(serial_ports):
            print(f"  {p}")
    else:
        print("  None found! Try: ls /dev/cu.usb*")

else:
    print("Windows detected.")
    print("This notebook uses stty/cat which require Linux/Mac.")
    print("On Windows without pyserial, please use PuTTY or TeraTerm.")
    print("Then paste the UART output into Cell 5 (manual parse mode).")

## 2. 配置串口参数

In [ ]:
# ========== MODIFY THESE ==========
SERIAL_PORT = "/dev/ttyUSB0"      # Linux: /dev/ttyUSB0, Mac: /dev/cu.usbserial-xxx
BAUD_RATE = 115200
CAPTURE_TIMEOUT = 30               # seconds to wait for UART data
CAPTURE_FILE = "/tmp/pico_uart.txt" # temp file to save captured output
GOLDEN_FILE = "fw_hex_batch/golden_results.txt"  # from prepare_picorv32_env.ipynb
# ==================================

print(f"Serial port:   {SERIAL_PORT}")
print(f"Baud rate:     {BAUD_RATE}")
print(f"Capture file:  {CAPTURE_FILE}")
print(f"Timeout:       {CAPTURE_TIMEOUT}s")
print(f"Golden file:   {GOLDEN_FILE} ({'EXISTS' if os.path.exists(GOLDEN_FILE) else 'NOT FOUND'})")

## 3. 配置串口 (stty) 并读取 UART

使用 `stty` 配置波特率，然后用后台 `cat` 捕获数据。

**操作顺序**:
1. 运行本 Cell
2. 在 PYNQ 上加载 bitstream (或按 BTN0 复位)
3. 等待 UART 数据捕获完成

In [ ]:
assert os.path.exists(SERIAL_PORT), f"{SERIAL_PORT} does not exist! Check USB-TTL connection."

# Step 1: Configure serial port with stty
stty_cmd = [
    "stty", "-F", SERIAL_PORT,
    str(BAUD_RATE),     # baud rate
    "cs8",              # 8 data bits
    "-cstopb",          # 1 stop bit
    "-parenb",          # no parity
    "raw",              # raw mode (no line processing)
    "-echo",            # no echo
    "-echoe",
    "-echok",
    "min", "0",         # non-blocking read
    "time", "10",       # 1 second timeout (in 1/10 sec)
]
# On Mac, stty uses different syntax
if platform.system() == "Darwin":
    stty_cmd = [
        "stty", "-f", SERIAL_PORT,
        str(BAUD_RATE), "cs8", "-cstopb", "-parenb",
        "raw", "-echo",
    ]

print("Configuring serial port...")
ret = subprocess.run(stty_cmd, capture_output=True, text=True)
if ret.returncode != 0:
    print(f"stty failed: {ret.stderr}")
    print("Try: sudo chmod 666 " + SERIAL_PORT)
else:
    print(f"  {SERIAL_PORT} configured: {BAUD_RATE} 8N1")

# Step 2: Flush old data
print("Flushing old data...")
try:
    with open(SERIAL_PORT, 'rb') as f:
        import select
        while select.select([f], [], [], 0.1)[0]:
            f.read(1024)
    print("  Buffer flushed")
except Exception as e:
    print(f"  Flush warning: {e} (may be OK)")

# Step 3: Capture UART output using cat in background
print(f"\nCapturing UART output to {CAPTURE_FILE}...")
print(f">>> Press BTN0 on PYNQ board NOW (or load bitstream) <<<")
print(f"Waiting up to {CAPTURE_TIMEOUT}s for data...\n")
print("-" * 60)

# Use dd with timeout to capture serial data
# Read character by character, stop when we see "=== Done ===" or timeout
uart_output = ""
start_time = time.time()
done = False

try:
    with open(SERIAL_PORT, 'rb', buffering=0) as serial_fd:
        import select
        buf = b""
        while time.time() - start_time < CAPTURE_TIMEOUT:
            ready, _, _ = select.select([serial_fd], [], [], 1.0)
            if ready:
                chunk = serial_fd.read(256)
                if chunk:
                    buf += chunk
                    # Process complete lines
                    while b"\n" in buf:
                        line_bytes, buf = buf.split(b"\n", 1)
                        line = line_bytes.decode(errors="ignore").rstrip("\r")
                        if line:
                            print(line)
                            uart_output += line + "\n"
                        if "=== Done ===" in line:
                            done = True
                            break
                    if done:
                        break

    elapsed = time.time() - start_time
    print("-" * 60)

    if done:
        print(f"\nInference complete! Elapsed: {elapsed:.1f}s")
    else:
        print(f"\nTimeout ({CAPTURE_TIMEOUT}s)! Received {len(uart_output)} chars")
        if not uart_output:
            print("\nNo data received. Checklist:")
            print("  [1] USB-TTL RXD <-- PMODA Pin 1 (Y18)?")
            print("  [2] GND connected?")
            print("  [3] PYNQ bitstream loaded? (LED1 heartbeat blinking?)")
            print("  [4] Pressed BTN0 after this Cell started?")
            print(f"  [5] Correct port? Permission? Try: sudo cat {SERIAL_PORT}")

    # Save to file
    if uart_output:
        with open(CAPTURE_FILE, "w") as f:
            f.write(uart_output)
        print(f"\nSaved to {CAPTURE_FILE}")

except PermissionError:
    print(f"Permission denied on {SERIAL_PORT}!")
    print(f"Run: sudo chmod 666 {SERIAL_PORT}")
    print(f"Or:  sudo usermod -aG dialout $USER  (then re-login)")
except Exception as e:
    print(f"Error: {e}")

## 4. 解析 UART 推理结果

In [ ]:
def parse_uart_output(text):
    """Parse PicoRV32 UART output into structured result."""
    result = {
        "predicted": None, "expected": None, "correct": None,
        "logits": [], "fc1_ok": False, "fc2_ok": False, "raw": text
    }
    for line in text.split("\n"):
        line = line.strip()
        if line.startswith("Predicted:"):
            try: result["predicted"] = int(line.split(":")[1].strip())
            except ValueError: pass
        elif line.startswith("Expected:"):
            try: result["expected"] = int(line.split(":")[1].strip())
            except ValueError: pass
        elif "CORRECT" in line:
            result["correct"] = True
        elif "WRONG" in line:
            result["correct"] = False
        elif line.startswith("Logits:"):
            nums = line.replace("Logits:", "").strip().split()
            result["logits"] = [int(x) for x in nums if x.lstrip("-").isdigit()]
        elif "FC1: done" in line:
            result["fc1_ok"] = True
        elif "FC2: computing" in line:
            result["fc2_ok"] = True
    return result

if uart_output:
    hw_result = parse_uart_output(uart_output)
    print("=" * 55)
    print("  PicoRV32 FPGA Inference Result (UART via stty+cat)")
    print("=" * 55)
    print(f"  FC1 (784->16, ReLU): {'OK' if hw_result['fc1_ok'] else 'N/A'}")
    print(f"  FC2 (16->10):        {'OK' if hw_result['fc2_ok'] else 'N/A'}")
    print(f"  Predicted: {hw_result['predicted']}")
    print(f"  Expected:  {hw_result['expected']}")
    status = 'CORRECT' if hw_result['correct'] else ('WRONG' if hw_result['correct'] is not None else 'UNKNOWN')
    print(f"  Result:    {status}")
    if hw_result["logits"]:
        print(f"  Logits:    {hw_result['logits']}")
        print(f"  ArgMax:    {np.argmax(hw_result['logits'])}")
    print("=" * 55)
else:
    hw_result = None
    print("No UART output to parse")

## 5. 手动粘贴模式 (备用)

如果自动捕获失败，可以从终端工具 (minicom / screen / PuTTY) 复制粘贴 UART 输出。

终端快速读取命令:
```bash
# Linux (需要 root 或 dialout 组)
stty -F /dev/ttyUSB0 115200 cs8 -cstopb -parenb raw -echo
cat /dev/ttyUSB0

# 或用 screen
screen /dev/ttyUSB0 115200

# Mac
screen /dev/cu.usbserial-xxx 115200
```

In [ ]:
# Paste UART output here (from minicom / PuTTY / screen / cat)
manual_output = """
=== CIM SoC RISC-V MNIST Inference ===
FC1: loading...
FC1: computing...
FC1: done
FC2: loading...
FC2: computing...

--- Result ---
Predicted: 7
Expected:  7
>>> CORRECT <<<
Logits: -16 -31 -1 18 -21 -1 -63 31 -13 6
=== Done ===
"""

# Set to True to parse from pasted text instead
USE_MANUAL = False

if USE_MANUAL and manual_output.strip():
    hw_result = parse_uart_output(manual_output)
    uart_output = manual_output
    print("Parsed from manual input:")
    print(f"  Predicted: {hw_result['predicted']}")
    print(f"  Expected:  {hw_result['expected']}")
    status = 'CORRECT' if hw_result['correct'] else 'WRONG'
    print(f"  Result:    {status}")
    if hw_result['logits']:
        print(f"  Logits:    {hw_result['logits']}")
else:
    print("Manual mode disabled. Set USE_MANUAL = True if auto-capture failed.")

## 6. 与 Golden Results 对比

使用 `fw_hex_batch/golden_results.txt` (由 `prepare_picorv32_env.ipynb` 生成)。

也可以对比 PYNQ 端 notebook 传回的 golden 结果。

In [ ]:
if hw_result and hw_result["predicted"] is not None and os.path.exists(GOLDEN_FILE):
    golden = {}
    with open(GOLDEN_FILE) as f:
        for line in f:
            if line.startswith("#"): continue
            parts = line.strip().split()
            if len(parts) >= 4:
                golden[int(parts[0])] = {
                    "label": int(parts[1]),
                    "pred": int(parts[2]),
                    "ok": int(parts[3])
                }
    print(f"Golden results loaded: {len(golden)} images")

    exp_label = hw_result["expected"]
    hw_pred = hw_result["predicted"]

    matched = [(idx, g) for idx, g in golden.items() if g["label"] == exp_label]

    if matched:
        idx, g = matched[0]
        print(f"\nMatched golden image #{idx}")
        print(f"  Golden:   label={g['label']}, pred={g['pred']}, {'correct' if g['ok'] else 'wrong'}")
        print(f"  Hardware: label={exp_label}, pred={hw_pred}")
        print()
        if hw_pred == g["pred"]:
            print("  +------------------------------------------+")
            print("  |  MATCH: Hardware == Golden (bit-exact!)   |")
            print("  +------------------------------------------+")
        else:
            print("  +------------------------------------------+")
            print("  |  MISMATCH: HW pred != Golden pred         |")
            print("  +------------------------------------------+")

        g_correct = sum(1 for v in golden.values() if v["ok"])
        print(f"\nGolden accuracy: {g_correct}/{len(golden)} ({100*g_correct/len(golden):.0f}%)")
    else:
        print(f"No golden record with label={exp_label}")

elif hw_result and hw_result["predicted"] is not None:
    print(f"Golden file not found: {GOLDEN_FILE}")
    print("Run prepare_picorv32_env.ipynb first.")
    print("Or check PYNQ-side notebook for golden results.")
else:
    print("No inference result to compare")

## 7. 与 PYNQ 端 Golden 结果交叉对比

如果 PYNQ 端 notebook (`picorv32_mlp_pynq_no_pyserial.ipynb`) 已生成
`pynq_golden_results.txt`，可以下载到 PC 进行交叉验证。

In [ ]:
PYNQ_GOLDEN = "pynq_golden_results.txt"  # Download from PYNQ Jupyter

if hw_result and hw_result["predicted"] is not None and os.path.exists(PYNQ_GOLDEN):
    pynq_golden = {}
    with open(PYNQ_GOLDEN) as f:
        for line in f:
            if line.startswith("#"): continue
            parts = line.strip().split()
            if len(parts) >= 3:
                pynq_golden[int(parts[0])] = {
                    "label": int(parts[1]),
                    "pred": int(parts[2]),
                }
    print(f"PYNQ golden loaded: {len(pynq_golden)} images")

    exp_label = hw_result["expected"]
    matched = [(k, v) for k, v in pynq_golden.items() if v["label"] == exp_label]
    if matched:
        idx, g = matched[0]
        print(f"  PYNQ golden #{idx}: label={g['label']}, pred={g['pred']}")
        print(f"  UART HW:          label={exp_label}, pred={hw_result['predicted']}")
        if hw_result['predicted'] == g['pred']:
            print("  >>> CROSS-CHECK MATCH <<<")
        else:
            print("  >>> CROSS-CHECK MISMATCH <<<")
else:
    print(f"{PYNQ_GOLDEN} not found.")
    print("Download from PYNQ Jupyter after running the PYNQ-side notebook.")

## 8. 保存结果

保存解析结果为文本文件，方便后续查阅或自动化脚本使用。

In [ ]:
RESULT_FILE = "pc_uart_result.txt"

if hw_result and hw_result["predicted"] is not None:
    with open(RESULT_FILE, "w") as f:
        f.write(f"# PC-side UART capture result\n")
        f.write(f"# Timestamp: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"predicted {hw_result['predicted']}\n")
        f.write(f"expected {hw_result['expected']}\n")
        f.write(f"correct {1 if hw_result['correct'] else 0}\n")
        if hw_result['logits']:
            f.write(f"logits {' '.join(str(x) for x in hw_result['logits'])}\n")
    print(f"Saved to {RESULT_FILE}")
    print("You can upload this to PYNQ for cross-verification.")
else:
    print("No result to save")

## 总结

本 notebook 在 **PC 上** 通过 `stty` + `open()` + `select` 读取 UART，**不需要 pyserial**。

```
PYNQ PL (PicoRV32+CIM) --UART--> USB-TTL --USB--> PC (/dev/ttyUSBx)
                                                      |
                                               stty + cat/select
                                                      |
                                               Parse + Compare
                                                      |
                        golden_results.txt <-----------+
                  (from prepare_picorv32_env.ipynb)
```

配合 PYNQ 端 `picorv32_mlp_pynq_no_pyserial.ipynb` 使用，实现双端验证。